# Indian Stock Analyzer · NSE

End-to-end analysis pipeline: market data → 3-category scoring → LLM interpretation → trader + investor verdict.

| | |
|---|---|
| **Stock** | RELIANCE.NS (Reliance Industries) |
| **Scoring** | 31 votes · 3 categories equally weighted · 12 technical · 15 fundamental · 4 India-specific |
| **India signals** | Promoter pledging · Stock PCR · Institutional holdings · Delivery % |
| **Model** | gpt-4.1-mini · structured JSON · separate trader and investor advice |
| **4 PM cell** | Post-close macro context: VIX · FII/DII flows · sector · commodities |
| **8 AM cell** | Pre-open global cues: US close · Asian markets · GIFT Nifty · S&P futures |

In [42]:
import os
import json
import warnings
import requests
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
import yfinance as yf
import pandas as pd
import ta
from openai import OpenAI

warnings.filterwarnings('ignore')
load_dotenv()

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

SYMBOL       = 'RELIANCE.NS'
COMPANY_NAME = 'Reliance Industries'

IST = timezone(timedelta(hours=5, minutes=30))

print('Setup complete')

Setup complete


## Market Data

One year of daily OHLCV — needed for MA200 (200 rows minimum) and the 52-week high/low range.
The bad-symbol test confirms yfinance fails silently with an empty DataFrame, not an exception.

In [43]:
ticker = yf.Ticker(SYMBOL)
df = ticker.history(period='1y')  # 1 year: needed for MA200 (200 rows) and 52-week range

print(f'Fetched {len(df)} rows for {SYMBOL}')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Columns: {list(df.columns)}')
df.tail(3)

Fetched 249 rows for RELIANCE.NS
Date range: 2025-04-22 to 2026-04-22
Columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-04-20 00:00:00+05:30,1363.199951,1373.000000,1352.800049,1363.300049,13614733,0.0,0.0
2026-04-21 00:00:00+05:30,1359.000000,1369.800049,1350.099976,1353.300049,27293629,0.0,0.0
2026-04-22 00:00:00+05:30,1350.300049,1366.000000,1349.099976,1362.099976,9525953,0.0,0.0


## Delivery %

NSE-specific conviction signal. Delivery % = shares held overnight ÷ total traded.

- **> 60%** — long-term investors dominating. If price is up: real buying. If price is down: real selling.
- **30–60%** — mix of short-term and long-term participants. Normal range.
- **< 30%** — mostly intraday / speculative trading. Price move has low conviction — treat with caution.

MarketsMojo flags delivery % prominently because it separates committed capital from noise.
Data source: NSE `securityWiseDP` endpoint (previous day, confirmed after market close).

In [44]:
import requests

_nse = requests.Session()
_nse.headers.update({
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36',
    'Accept': 'application/json',
    'Referer': 'https://www.nseindia.com/',
})
_nse.get('https://www.nseindia.com/', timeout=10)  # warm up session / get cookies

_resp = _nse.get(
    'https://www.nseindia.com/api/quote-equity',
    params={'symbol': SYMBOL.replace('.NS', ''), 'section': 'trade_info'},
    timeout=10,
)

delivery_pct  = None
delivery_date = None

if _resp.status_code == 200:
    _dp = _resp.json().get('securityWiseDP', {})
    delivery_pct  = _dp.get('deliveryToTradedQuantity')  # e.g. 63.05 (%)
    delivery_date = _dp.get('secWiseDelPosDate', 'previous day')

if delivery_pct is not None:
    label = (
        'strong conviction — committed capital' if delivery_pct > 60
        else 'speculative / intraday dominated — low conviction' if delivery_pct < 30
        else 'normal mix of short and long-term participants'
    )
    print(f'Delivery %  : {delivery_pct:.1f}%  ({delivery_date})')
    print(f'Reading     : {label}')
else:
    print('Delivery %  : unavailable (NSE session issue — will vote NEUTRAL in scoring)')


Delivery %  : 61.7%  (22-APR-2026 EOD)
Reading     : strong conviction — committed capital


In [45]:
bad_ticker = yf.Ticker('FAKESYMBOL.NS')
bad_df = bad_ticker.history(period='3mo')
label = 'empty DataFrame' if bad_df.empty else 'unexpectedly has data'
print(f'Bad symbol test: {len(bad_df)} rows — {label}')
# yfinance fails silently — no exception, just empty DataFrame
# Phase 2 adds explicit empty-check validation before downstream processing

$FAKESYMBOL.NS: possibly delisted; no price data found  (period=3mo) (Yahoo error = "No data found, symbol may be delisted")


Bad symbol test: 0 rows — empty DataFrame


## Technical Indicators

Twelve indicators across two groups — six directional signals, five oscillators, and one India-specific conviction signal.

| Indicator | What it measures | Signal |
|-----------|-----------------|--------|
| RSI (14) | Momentum — buyer vs seller dominance | < 30 oversold · > 70 overbought |
| MA20 / MA50 | Short and medium-term trend | Price above = uptrend |
| MA200 | Long-term trend — the institutional line | Above = funds buying · Below = funds avoiding |
| MACD | Momentum shift | Reliable only on normal+ volume |
| Volume ratio | Conviction behind today's move | > 1.5× confirms · < 0.5× dismissed |
| 52-week range | Position in the annual price range | > 75% from low = strong · < 25% = weak |
| Stochastic %K/%D | Short-term overbought / oversold + signal cross | < 20 · > 80 |
| Bollinger Bands | Price stretch relative to recent volatility | < 20% of band · > 80% of band |
| Williams %R | Proximity to recent high vs low | < −80 · > −20 |
| CCI | Deviation from the statistical average | < −100 · > +100 |
| **Delivery %** | **NSE-specific: committed capital vs intraday noise** | **> 60% = real conviction · < 30% = speculative** |

In [46]:
close = df['Close']
volume = df['Volume']

# Core trend & momentum
rsi = ta.momentum.RSIIndicator(close, window=14).rsi().iloc[-1]
ma20 = ta.trend.SMAIndicator(close, window=20).sma_indicator().iloc[-1]
ma50 = ta.trend.SMAIndicator(close, window=50).sma_indicator().iloc[-1]
ma200 = ta.trend.SMAIndicator(close, window=200).sma_indicator().iloc[-1]
macd_ind = ta.trend.MACD(close)
macd_bullish = macd_ind.macd().iloc[-1] > macd_ind.macd_signal().iloc[-1]

avg_vol_20 = volume.rolling(20).mean().iloc[-1]
volume_ratio = volume.iloc[-1] / avg_vol_20
current_price = close.iloc[-1]
high_52w = close.max()
low_52w = close.min()
pct_from_high = (current_price - high_52w) / high_52w * 100

# Additional oscillators for the scoring layer
stoch = ta.momentum.StochasticOscillator(df['High'], df['Low'], df['Close'])
stoch_k = stoch.stoch().iloc[-1]
stoch_d = stoch.stoch_signal().iloc[-1]

bb = ta.volatility.BollingerBands(close)
bb_upper = bb.bollinger_hband().iloc[-1]
bb_lower = bb.bollinger_lband().iloc[-1]
bb_pct = (current_price - bb_lower) / (bb_upper - bb_lower)  # 0=at lower, 1=at upper

wr = ta.momentum.WilliamsRIndicator(df['High'], df['Low'], df['Close']).williams_r().iloc[-1]
cci = ta.trend.CCIIndicator(df['High'], df['Low'], df['Close']).cci().iloc[-1]
pct_from_low = (current_price - low_52w) / (high_52w - low_52w) * 100  # 0=52w low, 100=52w high

# Plain-English labels
rsi_label = 'overbought — may pull back' if rsi > 70 else 'oversold — potential bounce' if rsi < 30 else 'neutral'
trend_ma50 = 'uptrend' if current_price > ma50 else 'downtrend'
trend_ma200 = 'long-term uptrend — institutions likely buying' if current_price > ma200 else 'long-term downtrend — many funds avoiding'
vol_label = 'unusually high — confirms price moves' if volume_ratio > 1.5 else 'unusually low — low conviction' if volume_ratio < 0.5 else 'normal'
w52_label = f'{abs(pct_from_high):.1f}% below 52w high' if pct_from_high < -1 else 'near 52-week high — strong demand zone'
macd_reliable = volume_ratio >= 0.5
macd_direction = 'bullish crossover' if macd_bullish else 'bearish crossover'
macd_label = macd_direction + ('' if macd_reliable else ' — LOW CONVICTION (volume too low, dismiss this signal)')

print(f'Current price : {current_price:,.0f}')
print(f'RSI (14)      : {rsi:.1f}  → {rsi_label}')
print(f'MA20 / MA50   : {ma20:,.0f} / {ma50:,.0f}  → MA20 {"above" if ma20 > ma50 else "below"} MA50')
print(f'MA200         : {ma200:,.0f}  → {trend_ma200}')
print(f'MACD          : {macd_label}')
print(f'Volume ratio  : {volume_ratio:.1f}x  → {vol_label}')
print(f'52-week       : Low {low_52w:,.0f} — High {high_52w:,.0f}  ({pct_from_low:.0f}% up from year-low)')
print(f'Stochastic    : %K {stoch_k:.1f}  %D {stoch_d:.1f}')
print(f'Bollinger     : price at {bb_pct:.0%} of band  (0%=lower, 100%=upper)')
print(f'Williams %R   : {wr:.1f}')
print(f'CCI           : {cci:.1f}')

# Delivery % direction signal (computed after delivery_pct is available)
today_return = (close.iloc[-1] - close.iloc[-2]) / close.iloc[-2] * 100
if delivery_pct is None:
    delivery_signal = 'NEUTRAL'
    delivery_label  = 'unavailable'
elif delivery_pct > 60:
    delivery_signal = 'BUY' if today_return >= 0 else 'SELL'
    delivery_label  = f'{delivery_pct:.0f}% — conviction {"buying" if today_return >= 0 else "selling"}'
elif delivery_pct < 30:
    delivery_signal = 'NEUTRAL'
    delivery_label  = f'{delivery_pct:.0f}% — speculative, low conviction'
else:
    delivery_signal = 'BUY' if today_return > 0.5 else 'SELL' if today_return < -0.5 else 'NEUTRAL'
    delivery_label  = f'{delivery_pct:.0f}% — normal'

print(f'Delivery    : {delivery_label}')


Current price : 1,362
RSI (14)      : 49.4  → neutral
MA20 / MA50   : 1,357 / 1,391  → MA20 below MA50
MA200         : 1,439  → long-term downtrend — many funds avoiding
MACD          : bullish crossover — LOW CONVICTION (volume too low, dismiss this signal)
Volume ratio  : 0.5x  → unusually low — low conviction
52-week       : Low 1,286 — High 1,592  (25% up from year-low)
Stochastic    : %K 76.4  %D 69.1
Bollinger     : price at 54% of band  (0%=lower, 100%=upper)
Williams %R   : -23.6
CCI           : 2.8
Delivery    : 62% — conviction buying


## Fundamentals

Fifteen metrics across six categories — growth, profitability, cash flow, valuation, financial health, and analyst consensus.
Each casts an independent BUY / NEUTRAL / SELL vote that feeds into the fundamental score (one of three equal-weight categories).

| Category | Metric | BUY | NEUTRAL | SELL |
|----------|--------|-----|---------|------|
| Growth | Revenue growth | > 15% | 5–15% | < 5% |
| Growth | Earnings growth | > 10% | 0–10% | < 0% |
| Profitability | Net margin | > 15% | 5–15% | < 5% |
| Profitability | EBITDA margin | > 20% | 10–20% | < 10% |
| Profitability | Return on assets | > 8% | 3–8% | < 3% |
| Cash flow | OCF quality ratio | > 0.9 | 0.5–0.9 | < 0.5 |
| Cash flow | Free cash flow | positive | — | negative |
| Valuation | Trailing P/E | < 18 | 18–30 | > 30 |
| Valuation | PEG ratio | < 1.0 | 1–2 | > 2 |
| Financial health | Debt / Equity | < 50% | 50–100% | > 100% |
| Financial health | Current ratio | > 2.0 | 1–2 | < 1.0 |
| Financial health | Interest coverage | > 3x | 1.5–3x | < 1.5x |
| Analyst | Consensus rating | < 2.0 | 2–3 | > 3 |
| Analyst | Upside to mean target | > 20% | 0–20% | < 0% |
| Analyst | Direction (3-month trend) | improving | stable | deteriorating |

> `debtToEquity` from yfinance for .NS stocks is in % scale (35.6 = 35.6% D/E ratio).
> OCF quality = operatingCashflow ÷ netIncome — values > 0.9 mean reported profits are backed by actual cash collected.

In [47]:
info = ticker.info  # fetched once, used for all fundamentals below

# ── Existing fields ──────────────────────────────────────────────────────────
revenue_growth  = info.get('revenueGrowth')
earnings_growth = info.get('earningsGrowth')
trailing_pe     = info.get('trailingPE')
forward_pe      = info.get('forwardPE')
peg_ratio       = info.get('pegRatio')
price_to_book   = info.get('priceToBook')
profit_margin   = info.get('profitMargins')
debt_to_equity  = info.get('debtToEquity')       # % scale for .NS (35.6 = 35.6% D/E)
promoter_stake  = info.get('heldPercentInsiders') # decimal (0.51 = 51%)
inst_holding    = info.get('heldPercentInstitutions')
analyst_count   = info.get('numberOfAnalystOpinions', 0)
analyst_mean    = info.get('recommendationMean')  # 1=Strong Buy, 5=Strong Sell
analyst_label   = info.get('averageAnalystRating', '')
target_low      = info.get('targetLowPrice')
target_mean     = info.get('targetMeanPrice')
target_high     = info.get('targetHighPrice')
analyst_upside  = (target_mean / current_price - 1) * 100 if target_mean else None
beta            = info.get('beta')

# ── New fields ───────────────────────────────────────────────────────────────
operating_cashflow = info.get('operatingCashflow')   # annual INR
free_cashflow      = info.get('freeCashflow')         # annual INR
ebitda_margins     = info.get('ebitdaMargins')        # decimal
return_on_assets   = info.get('returnOnAssets')       # decimal
current_ratio      = info.get('currentRatio')
net_income_ttm     = info.get('netIncomeToCommon')

# ── Fallback: financial statements when info fields are None ─────────────────
# yfinance ticker.info misses several fields for NSE stocks — pull from statements directly.
if any(v is None for v in [operating_cashflow, free_cashflow, return_on_assets, current_ratio]):
    def _stmt_val(df, keys):
        """Return first non-null value matching any of keys in a statement DataFrame."""
        if df is None or df.empty:
            return None
        for k in keys:
            if k in df.index:
                v = df.loc[k].iloc[0]
                if pd.notna(v):
                    return float(v)
        return None

    try:
        _cf = ticker.cashflow
        if operating_cashflow is None:
            operating_cashflow = _stmt_val(_cf, [
                'Operating Cash Flow', 'Cash From Operating Activities',
                'Total Cash From Operating Activities', 'Net Cash From Operating Activities',
            ])
        if free_cashflow is None:
            free_cashflow = _stmt_val(_cf, ['Free Cash Flow', 'FreeCashFlow'])
    except Exception:
        pass

    try:
        _bs = ticker.balance_sheet
        if current_ratio is None:
            _ca = _stmt_val(_bs, ['Current Assets', 'Total Current Assets'])
            _cl = _stmt_val(_bs, ['Current Liabilities', 'Total Current Liabilities'])
            if _ca and _cl and _cl > 0:
                current_ratio = _ca / _cl
        if return_on_assets is None:
            _total_assets = _stmt_val(_bs, ['Total Assets'])
            _net_inc = None
            try:
                _fin2 = ticker.financials
                _net_inc = _stmt_val(_fin2, [
                    'Net Income', 'Net Income Common Stockholders',
                    'Net Income From Continuing Operations',
                ])
            except Exception:
                pass
            if _total_assets and _net_inc and _total_assets > 0:
                return_on_assets = _net_inc / _total_assets
    except Exception:
        pass

# Cash flow quality — are reported profits backed by actual cash?
ocf_quality = (
    operating_cashflow / net_income_ttm
    if operating_cashflow and net_income_ttm and net_income_ttm > 0
    else None
)

# Interest coverage — can the company service its debt comfortably?
interest_coverage = None
try:
    fin = ticker.financials
    _ebit = next(
        (fin.loc[k].iloc[0] for k in ['EBIT', 'Operating Income', 'OperatingIncome'] if k in fin.index),
        None
    )
    _intexp = next(
        (fin.loc[k].iloc[0] for k in ['Interest Expense', 'InterestExpense'] if k in fin.index),
        None
    )
    if _ebit is not None and _intexp is not None and _intexp != 0:
        interest_coverage = abs(float(_ebit) / float(_intexp))
except Exception:
    pass

# Analyst direction — are recommendations improving or deteriorating over 3 months?
analyst_direction_vote  = 'NEUTRAL'
analyst_direction_label = 'stable'
try:
    recs = ticker.recommendations
    if recs is not None and not recs.empty:
        def _bull_frac(row):
            sb = int(row.get('strongBuy',  0) or 0)
            b  = int(row.get('buy',        0) or 0)
            h  = int(row.get('hold',       0) or 0)
            s  = int(row.get('sell',       0) or 0)
            ss = int(row.get('strongSell', 0) or 0)
            total = sb + b + h + s + ss
            return (sb + b) / total if total > 0 else 0.5
        current_bull = _bull_frac(recs.iloc[0])
        older_bull   = _bull_frac(recs.iloc[min(2, len(recs) - 1)])
        if current_bull > older_bull + 0.08:
            analyst_direction_vote  = 'BUY'
            analyst_direction_label = 'improving — more upgrades recently'
        elif current_bull < older_bull - 0.08:
            analyst_direction_vote  = 'SELL'
            analyst_direction_label = 'deteriorating — more downgrades recently'
except Exception:
    pass

# ── Display ──────────────────────────────────────────────────────────────────
def pct(v, d=1): return f'{v*100:.{d}f}%' if v is not None else 'N/A'
def num(v, d=1): return f'{v:.{d}f}'       if v is not None else 'N/A'
def cr(v):       return f'₹{v/1e7:,.0f} Cr' if v is not None else 'N/A'

print('── PROFITABILITY & GROWTH ──────────────────────────')
print(f'Revenue growth    : {pct(revenue_growth)}  (YoY)')
print(f'Earnings growth   : {pct(earnings_growth)}  (YoY)')
print(f'Net margin        : {pct(profit_margin)}')
print(f'EBITDA margin     : {pct(ebitda_margins)}')
print(f'Return on assets  : {pct(return_on_assets)}')
print()
print('── CASH FLOW ───────────────────────────────────────')
print(f'Operating CF      : {cr(operating_cashflow)}')
print(f'Free cash flow    : {cr(free_cashflow)}')
print(f'OCF quality ratio : {num(ocf_quality, 2)}  (>0.9 = profits backed by real cash)')
print()
print('── VALUATION ───────────────────────────────────────')
print(f'Trailing P/E      : {num(trailing_pe)}  |  Forward P/E : {num(forward_pe)}')
print(f'PEG ratio         : {num(peg_ratio, 2)}')
print(f'Price / Book      : {num(price_to_book, 2)}')
print()
print('── FINANCIAL HEALTH ────────────────────────────────')
print(f'Debt / Equity     : {num(debt_to_equity)}%  (yfinance % scale for .NS)')
print(f'Current ratio     : {num(current_ratio, 2)}  (>2 comfortable, <1 stressed)')
if interest_coverage:
    print(f'Interest coverage : {num(interest_coverage, 1)}x  (>3x healthy, <1.5x stressed)')
else:
    print('Interest coverage : N/A')
print()
print('── OWNERSHIP ───────────────────────────────────────')
print(f'Promoter stake    : {pct(promoter_stake)}')
print(f'Institutional     : {pct(inst_holding)}')
print(f'Beta              : {num(beta, 2)}')
print()
print('── ANALYST CONSENSUS ───────────────────────────────')
if analyst_count:
    print(f'Consensus         : {analyst_label}  ({analyst_count} analysts)')
    print(f'Direction (3m)    : {analyst_direction_label}')
    if target_mean:
        print(f'Targets           : Low {target_low:,.0f}  ·  Mean {target_mean:,.0f}  ·  High {target_high:,.0f}')
        print(f'Upside to mean    : {analyst_upside:.1f}%')
        if target_low and target_low > current_price:
            print(f'  ★ Even the most bearish analyst targets {target_low:,.0f} — above current price')
else:
    print('Consensus         : N/A')

── PROFITABILITY & GROWTH ──────────────────────────
Revenue growth    : 10.4%  (YoY)
Earnings growth   : 0.6%  (YoY)
Net margin        : 8.1%
EBITDA margin     : 16.4%
Return on assets  : 3.6%

── CASH FLOW ───────────────────────────────────────
Operating CF      : ₹178,703 Cr
Free cash flow    : ₹38,736 Cr
OCF quality ratio : 2.15  (>0.9 = profits backed by real cash)

── VALUATION ───────────────────────────────────────
Trailing P/E      : 22.2  |  Forward P/E : 20.8
PEG ratio         : 0.82
Price / Book      : 2.10

── FINANCIAL HEALTH ────────────────────────────────
Debt / Equity     : 35.7%  (yfinance % scale for .NS)
Current ratio     : 1.10  (>2 comfortable, <1 stressed)
Interest coverage : 5.8x  (>3x healthy, <1.5x stressed)

── OWNERSHIP ───────────────────────────────────────
Promoter stake    : 51.0%
Institutional     : 27.9%
Beta              : 0.22

── ANALYST CONSENSUS ───────────────────────────────
Consensus         : 1.4 - Strong Buy  (33 analysts)
Direction (3m)   

## India-Specific Signals

NSE-only data points that no global platform incorporates into automated scoring.

| Signal | BUY | NEUTRAL | SELL |
|--------|-----|---------|------|
| Promoter holding % | 35–70% (aligned, float healthy) | 20–35% or 70–80% | <20% or >80% |
| Promoter pledge % | <5% (no financial stress) | 5–25% | >25% (forced-sell risk) |
| Stock PCR | <0.7 (calls > puts) | 0.7–1.2 | >1.2 (heavy put hedging) |
| Institutional holdings | >30% (smart money present) | 10–30% | <10% (institutions avoiding) |

In [48]:
# ── India-Specific Signals ──────────────────────────────────────────────────
# Reuses the NSE session from cell-delivery if available; creates a fresh one otherwise.
# Always re-warms cookies — NSE sessions expire quickly between cells.

if '_nse' not in globals():
    _nse = requests.Session()
    _nse.headers.update({
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36',
        'Accept': 'application/json',
        'Referer': 'https://www.nseindia.com/',
    })

# Always re-warm cookies before NSE API calls
_nse.get('https://www.nseindia.com/', timeout=10)

NSE_SYM = SYMBOL.replace('.NS', '')

# ── Stock PCR from option chain ──────────────────────────────────────────────
stock_pcr = None
try:
    _oc = _nse.get(
        'https://www.nseindia.com/api/option-chain-equities',
        params={'symbol': NSE_SYM},
        timeout=15,
    )
    if _oc.status_code == 200:
        _oc_data = _oc.json().get('records', {}).get('data', [])
        _ce_oi   = sum(r.get('CE', {}).get('openInterest', 0) for r in _oc_data if 'CE' in r)
        _pe_oi   = sum(r.get('PE', {}).get('openInterest', 0) for r in _oc_data if 'PE' in r)
        if _ce_oi > 0:
            stock_pcr = _pe_oi / _ce_oi
except Exception:
    pass

# ── Promoter pledging from NSE ───────────────────────────────────────────────
promoter_pledge_pct = None
try:
    _pledge = _nse.get(
        'https://www.nseindia.com/api/corporate-pledgedata',
        params={'index': 'equities', 'symbol': NSE_SYM},
        timeout=10,
    )
    if _pledge.status_code == 200:
        _pdata = _pledge.json()
        _rows  = _pdata if isinstance(_pdata, list) else _pdata.get('data', [])
        if _rows:
            for _key in ['pledgedSharesPerc', 'pledgePerc', 'pledgePercent',
                         'promoterAndPromoterGroupPercentagePledge', 'promoterPledgePerc']:
                if _key in _rows[0]:
                    promoter_pledge_pct = float(_rows[0][_key])
                    break
except Exception:
    pass

# ── Display ──────────────────────────────────────────────────────────────────
print('── INDIA-SPECIFIC ──────────────────────────────────')

if stock_pcr is not None:
    pcr_label = (
        'bullish — calls > puts' if stock_pcr < 0.7
        else 'bearish — puts > calls, institutions hedging' if stock_pcr > 1.2
        else 'neutral'
    )
    print(f'Stock PCR        : {stock_pcr:.2f}  ({pcr_label})')
else:
    print('Stock PCR        : unavailable  (voting NEUTRAL)')

if promoter_pledge_pct is not None:
    pledge_label = (
        'minimal — no financial stress' if promoter_pledge_pct < 5
        else 'moderate — monitor for escalation' if promoter_pledge_pct < 25
        else 'HIGH — forced-selling risk if stock corrects'
    )
    print(f'Promoter pledge  : {promoter_pledge_pct:.1f}%  ({pledge_label})')
else:
    print('Promoter pledge  : unavailable  (voting NEUTRAL)')

_ph = promoter_stake * 100 if promoter_stake else None
_ih = inst_holding  * 100 if inst_holding  else None
print(f'Promoter holding : {_ph:.1f}%' if _ph else 'Promoter holding : N/A')
print(f'Institutional    : {_ih:.1f}%' if _ih else 'Institutional    : N/A')

── INDIA-SPECIFIC ──────────────────────────────────
Stock PCR        : unavailable  (voting NEUTRAL)
Promoter pledge  : unavailable  (voting NEUTRAL)
Promoter holding : 51.0%
Institutional    : 27.9%


## Signal Scoring

Each indicator casts an independent BUY / NEUTRAL / SELL vote using fixed rules.
31 votes across 3 categories — each category is scored independently from −1 to +1, then averaged equally (33/33/33 weight).

| Category | Votes | What it captures |
|----------|-------|-----------------|
| Technical | 12 | Price action, momentum, volume conviction |
| Fundamental | 15 | Growth, profitability, cash flow, valuation, financial health, analyst consensus |
| India-specific | 4 | Promoter pledging, stock PCR, institutional holdings, promoter stake |

The LLM receives all three category scores and the breakdown — it can see and explain any disagreement between price action, business fundamentals, and India-market positioning.

In [49]:
# ── Technical votes (12) ─────────────────────────────────────────────────────
votes = {}
votes['RSI']            = 'BUY' if rsi < 30 else 'SELL' if rsi > 70 else 'NEUTRAL'
votes['Price_vs_MA50']  = 'BUY' if current_price > ma50  else 'SELL'
votes['Price_vs_MA200'] = 'BUY' if current_price > ma200 else 'SELL'
votes['MA20_vs_MA50']   = 'BUY' if ma20 > ma50           else 'SELL'
votes['MACD']           = ('BUY' if macd_bullish else 'SELL') if macd_reliable else 'NEUTRAL'
votes['Delivery_Pct']   = delivery_signal
votes['Stoch_K_Level']  = 'BUY' if stoch_k < 20 else 'SELL' if stoch_k > 80 else 'NEUTRAL'
votes['Stoch_Signal']   = 'BUY' if stoch_k > stoch_d else 'SELL'
votes['Bollinger']      = 'BUY' if bb_pct < 0.2 else 'SELL' if bb_pct > 0.8 else 'NEUTRAL'
votes['Williams_R']     = 'BUY' if wr < -80 else 'SELL' if wr > -20 else 'NEUTRAL'
votes['CCI']            = 'BUY' if cci < -100 else 'SELL' if cci > 100 else 'NEUTRAL'
votes['52w_Position']   = 'BUY' if pct_from_low > 75 else 'SELL' if pct_from_low < 25 else 'NEUTRAL'

# ── Fundamental votes (15) ───────────────────────────────────────────────────
def fvote(val, buy_thresh, sell_thresh, higher_is_better=True):
    if val is None: return 'NEUTRAL'
    if higher_is_better:
        return 'BUY' if val >= buy_thresh else 'SELL' if val <= sell_thresh else 'NEUTRAL'
    else:
        return 'BUY' if val <= buy_thresh else 'SELL' if val >= sell_thresh else 'NEUTRAL'

fund_votes = {}
# Growth
fund_votes['Revenue_Growth']    = fvote(revenue_growth,   0.15, 0.05)
fund_votes['Earnings_Growth']   = fvote(earnings_growth,  0.10, 0.00)
# Profitability
fund_votes['Net_Margin']        = fvote(profit_margin,    0.15, 0.05)
fund_votes['EBITDA_Margin']     = fvote(ebitda_margins,   0.20, 0.10)
fund_votes['Return_on_Assets']  = fvote(return_on_assets, 0.08, 0.03)
# Cash flow
fund_votes['OCF_Quality']       = fvote(ocf_quality,       0.9,  0.5)
fund_votes['Free_Cash_Flow']    = (
    'BUY'  if free_cashflow and free_cashflow > 0
    else 'SELL' if free_cashflow and free_cashflow < 0
    else 'NEUTRAL'
)
# Valuation
fund_votes['Trailing_PE']       = fvote(trailing_pe,  18.0, 30.0, higher_is_better=False)
fund_votes['PEG_Ratio']         = fvote(peg_ratio,     1.0,  2.0, higher_is_better=False)
# Financial health
fund_votes['Debt_Equity']       = fvote(debt_to_equity,  50.0, 100.0, higher_is_better=False)
fund_votes['Current_Ratio']     = fvote(current_ratio,    2.0,   1.0)
fund_votes['Interest_Coverage'] = fvote(interest_coverage, 3.0,  1.5)
# Analyst
fund_votes['Analyst_Rating']    = fvote(analyst_mean,  2.0, 3.0, higher_is_better=False)
fund_votes['Analyst_Upside']    = fvote(analyst_upside, 20.0, 0.0)
fund_votes['Analyst_Direction'] = analyst_direction_vote

# ── India-specific votes (4) ─────────────────────────────────────────────────
# Safely read from cell-india output; defaults to NEUTRAL if that cell wasn't run.
_stock_pcr       = globals().get('stock_pcr')
_pledge_pct      = globals().get('promoter_pledge_pct')
_ph              = promoter_stake * 100 if promoter_stake else None
_ih              = inst_holding   * 100 if inst_holding   else None

india_votes = {}
# Promoter holding: 35–70% ideal (aligned + float healthy)
if   _ph is None:          india_votes['Promoter_Holding'] = 'NEUTRAL'
elif 35 <= _ph <= 70:      india_votes['Promoter_Holding'] = 'BUY'
elif _ph < 20:             india_votes['Promoter_Holding'] = 'SELL'
else:                      india_votes['Promoter_Holding'] = 'NEUTRAL'

# Promoter pledging: <5% safe, >25% red flag
if   _pledge_pct is None:  india_votes['Promoter_Pledge'] = 'NEUTRAL'
elif _pledge_pct < 5:      india_votes['Promoter_Pledge'] = 'BUY'
elif _pledge_pct > 25:     india_votes['Promoter_Pledge'] = 'SELL'
else:                      india_votes['Promoter_Pledge'] = 'NEUTRAL'

# Stock PCR: <0.7 bullish, >1.2 bearish
if   _stock_pcr is None:   india_votes['Stock_PCR'] = 'NEUTRAL'
elif _stock_pcr < 0.7:     india_votes['Stock_PCR'] = 'BUY'
elif _stock_pcr > 1.2:     india_votes['Stock_PCR'] = 'SELL'
else:                      india_votes['Stock_PCR'] = 'NEUTRAL'

# Institutional holdings: >30% = smart money present, <10% = avoiding
if   _ih is None:          india_votes['Institutional_Holdings'] = 'NEUTRAL'
elif _ih > 30:             india_votes['Institutional_Holdings'] = 'BUY'
elif _ih < 10:             india_votes['Institutional_Holdings'] = 'SELL'
else:                      india_votes['Institutional_Holdings'] = 'NEUTRAL'

# ── Scoring ───────────────────────────────────────────────────────────────────
def tally(d):
    b = sum(1 for v in d.values() if v == 'BUY')
    s = sum(1 for v in d.values() if v == 'SELL')
    n = sum(1 for v in d.values() if v == 'NEUTRAL')
    return b, s, n

def category_score(b, s, total):
    return (b - s) / total if total > 0 else 0.0

def grade(score):
    if   score >=  0.30: return 'STRONG BUY'
    elif score >=  0.08: return 'LEAN BUY'
    elif score <= -0.30: return 'STRONG SELL'
    elif score <= -0.08: return 'LEAN SELL'
    else:                return 'NEUTRAL'

tb, ts, tn = tally(votes)
fb, fs, fn = tally(fund_votes)
ib, is_, in_ = tally(india_votes)

tech_score    = category_score(tb, ts, len(votes))
fund_score    = category_score(fb, fs, len(fund_votes))
india_score   = category_score(ib, is_, len(india_votes))
combined_score = (tech_score + fund_score + india_score) / 3  # equal 3-way

tech_grade    = grade(tech_score)
fund_grade    = grade(fund_score)
india_grade   = grade(india_score)
combined_grade = grade(combined_score)

total_votes = len(votes) + len(fund_votes) + len(india_votes)

print(f'Technical votes ({len(votes)}):')
for name, v in votes.items():
    icon = '↑' if v == 'BUY' else '↓' if v == 'SELL' else '→'
    print(f'  {icon}  {name:24s}  {v}')
print(f'  ⟹  {tech_grade}: {tb}B · {tn}N · {ts}S  (score {tech_score:+.2f})')
print()

print(f'Fundamental votes ({len(fund_votes)}):')
for name, v in fund_votes.items():
    icon = '↑' if v == 'BUY' else '↓' if v == 'SELL' else '→'
    print(f'  {icon}  {name:24s}  {v}')
print(f'  ⟹  {fund_grade}: {fb}B · {fn}N · {fs}S  (score {fund_score:+.2f})')
print()

print(f'India-specific votes ({len(india_votes)}):')
for name, v in india_votes.items():
    icon = '↑' if v == 'BUY' else '↓' if v == 'SELL' else '→'
    print(f'  {icon}  {name:24s}  {v}')
print(f'  ⟹  {india_grade}: {ib}B · {in_}N · {is_}S  (score {india_score:+.2f})')
print()

# Summary strings used by LLM and format cells
tech_summary    = f'{tech_grade}: {tb}B · {tn}N · {ts}S (score {tech_score:+.2f})'
fund_summary    = f'{fund_grade}: {fb}B · {fn}N · {fs}S (score {fund_score:+.2f})'
india_summary   = f'{india_grade}: {ib}B · {in_}N · {is_}S (score {india_score:+.2f})'
combined_summary = (
    f'{combined_grade}  '
    f'(tech {tech_score:+.2f} · fund {fund_score:+.2f} · india {india_score:+.2f} · avg {combined_score:+.2f})'
)
all_b = tb + fb + ib
all_s = ts + fs + is_
all_n = tn + fn + in_

print(f'COMBINED ({total_votes} votes, 3 categories equal weight)')
print(f'  ⟹  {combined_summary}')
print()
print('Neutrals are abstentions. Each category contributes equally (33/33/33).')

Technical votes (12):
  →  RSI                       NEUTRAL
  ↓  Price_vs_MA50             SELL
  ↓  Price_vs_MA200            SELL
  ↓  MA20_vs_MA50              SELL
  →  MACD                      NEUTRAL
  ↑  Delivery_Pct              BUY
  →  Stoch_K_Level             NEUTRAL
  ↑  Stoch_Signal              BUY
  →  Bollinger                 NEUTRAL
  →  Williams_R                NEUTRAL
  →  CCI                       NEUTRAL
  ↓  52w_Position              SELL
  ⟹  LEAN SELL: 2B · 6N · 4S  (score -0.17)

Fundamental votes (15):
  →  Revenue_Growth            NEUTRAL
  →  Earnings_Growth           NEUTRAL
  →  Net_Margin                NEUTRAL
  →  EBITDA_Margin             NEUTRAL
  →  Return_on_Assets          NEUTRAL
  ↑  OCF_Quality               BUY
  ↑  Free_Cash_Flow            BUY
  →  Trailing_PE               NEUTRAL
  ↑  PEG_Ratio                 BUY
  ↑  Debt_Equity               BUY
  →  Current_Ratio             NEUTRAL
  ↑  Interest_Coverage         BUY
  ↑  Analyst_

## News

Stock-specific headlines via `ticker.news` (Yahoo Finance editorial feed).
Title alone is not enough — a two-sentence summary snippet is included so the LLM understands severity, not just topic.

Tavily is reserved for one broad market themes search per day in the Phase 2 screener (1,000 credits/month budget).

In [50]:
news_raw = ticker.news  # ticker already defined in Task 1.2

news_items = []
for item in news_raw[:6]:
    content = item.get('content', {})
    title = item.get('title') or content.get('title', '')
    summary = content.get('summary', '') or item.get('description', '')
    publisher = content.get('provider', {}).get('displayName', '') or item.get('publisher', '')
    if title:
        news_items.append({'title': title, 'summary': summary[:200], 'publisher': publisher})

news_items = news_items[:3]
if not news_items:
    news_items = [{'title': 'No recent news found', 'summary': '', 'publisher': ''}]

# Format: headline + summary snippet — this is what goes into the LLM prompt
# Headline alone is not enough: "Reliance executive arrested" vs the summary telling you it's
# a bribery probe are very different signals. Summary adds severity without bloating tokens.
news_context = chr(10).join(
    f'- [{item["publisher"]}] {item["title"]}' +
    (f'\n  {item["summary"]}' if item['summary'] else '')
    for item in news_items
)

print(f'News ({len(news_items)} items):')
print(news_context)

News (3 items):
- [Reuters] India arrests officials at aviation regulator, Reliance in drone bribery probe
  NEW DELHI, April 20 (Reuters) - India's federal crime fighting agency said it has arrested an official from the country's aviation regulator and a Reliance Industries executive on allegations of
- [Simply Wall St.] How The Story On Reliance Industries (NSEI:RELIANCE) Is Shifting As Analysts Rework Fair Value
  Reliance Industries has seen its modelled fair value edge from ₹1,719.94 to ₹1,732.03, a modest adjustment that still matters if you are tracking where analysts think the stock should sit. The shift e
- [TechCrunch] HBO Max comes to India via exclusive JioHotstar deal
  HBO Max will be available to JioHotstar subscribers as an add-on starting at ₹49 (about $0.50) per month and would feature content from HBO, Max Originals, Warner Bros. Pictures, Warner Bros. Televisi


## Corporate Events

NSE filings — earnings dates, dividends, splits — do not reliably surface in news feeds.
`ticker.calendar`, `ticker.dividends`, and `ticker.actions` fetch them directly.

Upcoming earnings within 5 days hard-cap the LLM confidence at 0.50 regardless of other signals.

In [51]:
from datetime import date

cal = ticker.calendar
divs = ticker.dividends.tail(4)
actions = ticker.actions

corp_lines = []

# Upcoming earnings — most important: pre-earnings = high uncertainty
if cal and 'Earnings Date' in cal:
    earnings_dates = cal['Earnings Date']
    if isinstance(earnings_dates, list) and earnings_dates:
        next_earnings = earnings_dates[0]
        days_away = (next_earnings - date.today()).days
        if 0 <= days_away <= 5:
            corp_lines.append(f'EARNINGS IN {days_away} DAYS ({next_earnings}) — results imminent, expect volatility')
        elif 0 < days_away <= 30:
            corp_lines.append(f'Earnings on {next_earnings} ({days_away} days away)')

# Dividend history — is it growing, stable, or cut?
if not divs.empty:
    recent = [(str(d.date()), round(float(v), 2)) for d, v in divs.items()]
    values = [v for _, v in recent]
    trend = 'growing' if values[-1] > values[0] else 'declining' if values[-1] < values[0] else 'stable'
    summary = ', '.join(f'Rs.{v} ({d})' for d, v in recent[-3:])
    corp_lines.append(f'Dividend history (last 3): {summary} — trend: {trend}')

# Upcoming ex-dividend date
ex_div = cal.get('Ex-Dividend Date') if cal else None
if ex_div:
    days_away = (ex_div - date.today()).days
    if 0 <= days_away <= 30:
        corp_lines.append(f'Ex-dividend date in {days_away} days ({ex_div})')

# Recent stock splits — distort all price-based indicators
if 'Stock Splits' in actions.columns:
    recent_splits = actions[actions['Stock Splits'] > 0].tail(2)
    if not recent_splits.empty:
        for d, row in recent_splits.iterrows():
            corp_lines.append(f'Stock split {int(row["Stock Splits"])}:1 on {d.date()} — indicators adjusted')

corporate_context = chr(10).join(f'- {l}' for l in corp_lines)
if not corp_lines:
    corporate_context = '- No upcoming corporate events'

print('Corporate events:')
print(corporate_context)

Corporate events:
- EARNINGS IN 1 DAYS (2026-04-24) — results imminent, expect volatility
- Dividend history (last 3): Rs.4.5 (2023-08-21), Rs.5.0 (2024-08-19), Rs.5.5 (2025-08-14) — trend: growing
- Stock split 2:1 on 2017-09-07 — indicators adjusted
- Stock split 2:1 on 2024-10-28 — indicators adjusted


## Macro Context  ·  4 PM Post-Close

Run independently after NSE closes (3:30 PM IST). Fetches India market context that surrounds the stock-specific signal.

At 4 PM IST: Nifty and sector closes are final · FII/DII provisional data is published · Commodities (Brent, Gold, USD/INR) are live 24h markets. US market hasn't opened yet — that data comes in the 8 AM cell.

In [52]:
# ── 4 PM Macro Context ──────────────────────────────────────────────────────
# Run after NSE closes (3:30 PM IST). All India data is settled.
# Standalone — can be re-run independently without re-running the full analysis.
# Output stored in `macro_context` dict; LLM cell reads it automatically.

now_ist = datetime.now(IST)

def _yf_hist(sym, period='5d'):
    try:
        h = yf.Ticker(sym).history(period=period)
        return h if not h.empty else pd.DataFrame()
    except Exception:
        return pd.DataFrame()

def _day_ret(h):
    if len(h) >= 2:
        return (h['Close'].iloc[-1] - h['Close'].iloc[-2]) / h['Close'].iloc[-2] * 100
    return None

def _last(h):
    return h['Close'].iloc[-1] if not h.empty else None

def _arr(v): return '↑' if v and v >= 0 else '↓'

# India VIX and Nifty
vix_h     = _yf_hist('^INDIAVIX')
india_vix = _last(vix_h)
nifty_h   = _yf_hist('^NSEI')
nifty_ret = _day_ret(nifty_h)
nifty_lvl = _last(nifty_h)

# Sector index for this stock
SECTOR_MAP = {
    'RELIANCE.NS':   ('^CNXEnergy',   'Nifty Energy'),
    'TCS.NS':        ('^CNXInfotech', 'Nifty IT'),
    'HDFCBANK.NS':   ('^NSEBANK',     'Bank Nifty'),
    'INFY.NS':       ('^CNXInfotech', 'Nifty IT'),
    'WIPRO.NS':      ('^CNXInfotech', 'Nifty IT'),
    'ONGC.NS':       ('^CNXEnergy',   'Nifty Energy'),
    'TATAMOTORS.NS': ('^CNXAUTO',    'Nifty Auto'),
    'MARUTI.NS':     ('^CNXAUTO',    'Nifty Auto'),
    'SUNPHARMA.NS':  ('^CNXPHARMA',  'Nifty Pharma'),
    'DRREDDY.NS':    ('^CNXPHARMA',  'Nifty Pharma'),
}
sector_sym, sector_name = SECTOR_MAP.get(SYMBOL, ('^NSEI', 'Nifty 50'))
sect_h   = _yf_hist(sector_sym)
sect_ret = _day_ret(sect_h)

# Commodities (24h markets — always live)
brent_h     = _yf_hist('BZ=F')
brent_price = _last(brent_h)
brent_ret   = _day_ret(brent_h)
gold_price  = _last(_yf_hist('GC=F'))
usdinr      = _last(_yf_hist('INR=X'))

# FII/DII provisional flows from NSE
fii_net = None; dii_net = None
try:
    if '_nse' not in globals():
        _nse = requests.Session()
        _nse.headers.update({
            'User-Agent': 'Mozilla/5.0',
            'Accept': 'application/json',
            'Referer': 'https://www.nseindia.com/',
        })
        _nse.get('https://www.nseindia.com/', timeout=10)
    _fii_r = _nse.get('https://www.nseindia.com/api/fiidiiTradeReact', timeout=10)
    if _fii_r.status_code == 200:
        for item in _fii_r.json():
            cat = str(item.get('category', '')).upper()
            for key in ['netValue', 'net', 'netTurnover', 'net_value']:
                if key in item and item[key] is not None:
                    val = float(str(item[key]).replace(',', ''))
                    if 'FII' in cat or 'FPI' in cat: fii_net = val
                    elif 'DII' in cat: dii_net = val
                    break
except Exception:
    pass

# Nifty option chain PCR (market-wide sentiment)
nifty_pcr = None
try:
    _nr = _nse.get(
        'https://www.nseindia.com/api/option-chain-indices',
        params={'symbol': 'NIFTY'}, timeout=10
    )
    if _nr.status_code == 200:
        _d = _nr.json().get('records', {}).get('data', [])
        _ce = sum(r.get('CE', {}).get('openInterest', 0) for r in _d if 'CE' in r)
        _pe = sum(r.get('PE', {}).get('openInterest', 0) for r in _d if 'PE' in r)
        nifty_pcr = _pe / _ce if _ce > 0 else None
except Exception:
    pass

# ── Display ──────────────────────────────────────────────────────────────────
SEP64 = '━' * 64
print(SEP64)
print(f'MACRO CONTEXT  ·  {now_ist.strftime("%d %b %Y  %H:%M IST")}  ·  Run at 4 PM post-close')
print(SEP64)
print('INDIA')
if nifty_lvl:
    print(f'  Nifty 50     : {nifty_lvl:,.0f}  {_arr(nifty_ret)}{abs(nifty_ret):.2f}%' if nifty_ret is not None else f'  Nifty 50     : {nifty_lvl:,.0f}')
if sect_ret is not None:
    vs = f'  (vs Nifty {_arr(nifty_ret)}{abs(nifty_ret):.2f}%)' if nifty_ret is not None else ''
    print(f'  {sector_name:14s}: {_arr(sect_ret)}{abs(sect_ret):.2f}%{vs}')
if india_vix:
    vix_lbl = 'fear — cautious' if india_vix > 20 else 'low fear — calm' if india_vix < 14 else 'normal'
    print(f'  India VIX    : {india_vix:.1f}  ({vix_lbl})')
if fii_net is not None:
    print(f'  FII net      : ₹{abs(fii_net):,.0f} Cr  {_arr(fii_net)} ({"buying" if fii_net >= 0 else "selling"})')
if dii_net is not None:
    print(f'  DII net      : ₹{abs(dii_net):,.0f} Cr  {_arr(dii_net)} ({"buying" if dii_net >= 0 else "selling"})')
if nifty_pcr:
    print(f'  Nifty PCR    : {nifty_pcr:.2f}  ({"bearish hedge" if nifty_pcr > 1 else "bullish positioning"})')
print()
print('COMMODITIES')
if brent_price:
    print(f'  Brent crude  : ${brent_price:.1f}  {_arr(brent_ret)}{abs(brent_ret):.1f}%' if brent_ret else f'  Brent crude  : ${brent_price:.1f}')
if gold_price:   print(f'  Gold         : ${gold_price:,.0f}')
if usdinr:       print(f'  USD/INR      : ₹{usdinr:.2f}')
print()
reads = []
if nifty_ret and nifty_ret > 1:     reads.append('broad market strong')
if nifty_ret and nifty_ret < -1:    reads.append('broad market weak')
if fii_net and fii_net < -1000:     reads.append(f'heavy FII selling ₹{abs(fii_net):,.0f} Cr')
if fii_net and fii_net > 1000:      reads.append(f'strong FII buying ₹{fii_net:,.0f} Cr')
if india_vix and india_vix > 20:    reads.append(f'elevated fear (VIX {india_vix:.0f})')
if brent_price and brent_price > 90: reads.append(f'elevated oil ${brent_price:.0f} — inflation risk')
if sect_ret is not None and nifty_ret is not None and abs(sect_ret - nifty_ret) > 1:
    reads.append(f'{sector_name} {"outperforming" if sect_ret > nifty_ret else "underperforming"} market')
print('Market read  :  ' + (' · '.join(reads).capitalize() if reads else 'Neutral conditions'))
print(SEP64)

macro_context = {
    'nifty_return_pct':  nifty_ret,
    'sector_return_pct': sect_ret,
    'sector_name':       sector_name,
    'india_vix':         india_vix,
    'fii_net_cr':        fii_net,
    'dii_net_cr':        dii_net,
    'brent_price':       brent_price,
    'gold_price':        gold_price,
    'usdinr':            usdinr,
    'nifty_pcr':         nifty_pcr,
}
print('\nmacro_context stored — LLM cell will include this automatically.')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MACRO CONTEXT  ·  23 Apr 2026  08:15 IST  ·  Run at 4 PM post-close
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INDIA
  Nifty 50     : 24,378  ↓0.81%
  Nifty Energy  : ↑1.42%  (vs Nifty ↓0.81%)
  India VIX    : 18.3  (normal)
  FII net      : ₹2,078 Cr  ↓ (selling)
  DII net      : ₹1,048 Cr  ↓ (selling)

COMMODITIES
  Brent crude  : $103.0  ↑4.5%
  Gold         : $4,729
  USD/INR      : ₹93.78

Market read  :  Heavy fii selling ₹2,078 cr · elevated oil $103 — inflation risk · nifty energy outperforming market
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

macro_context stored — LLM cell will include this automatically.


## LLM Analysis

The pre-computed score, fundamental context, corporate events, and news are sent to gpt-4.1-mini.
Token count prints after the call — use it to project cost at 30 stocks (Phase 2 scale).

In [53]:
def _safe_pct(v): return f'{v*100:.1f}%' if v is not None else 'N/A'
def _safe_num(v, d=1): return f'{v:.{d}f}' if v is not None else 'N/A'

# ── Build user message ────────────────────────────────────────────────────────
lines = [
    f'Stock: {SYMBOL} ({COMPANY_NAME})',
    f'Current price: ₹{current_price:,.0f}',
    f'Beta: {_safe_num(beta, 2)} (stock volatility vs Nifty)',
    '',
    f'COMBINED VERDICT ({total_votes} indicators, 3 categories equal weight):',
    f'  {combined_summary}',
    f'  Technical      ({len(votes):2d}): {tech_summary}',
    f'  Fundamental    ({len(fund_votes):2d}): {fund_summary}',
    f'  India-specific ({len(india_votes):2d}): {india_summary}',
    '',
    'Technical context:',
    f'  RSI {rsi:.1f} ({rsi_label})',
    f'  {trend_ma200}',
    f'  Stochastic %K {stoch_k:.1f} vs %D {stoch_d:.1f} · Bollinger at {bb_pct:.0%} of band',
    f'  Volume ratio: {volume_ratio:.1f}x average · MACD: {macd_label}',
    f'  52-week position: {pct_from_low:.0f}% up from year-low ({w52_label})',
    f'  Delivery %: {delivery_label}',
    '',
    'Fundamental context:',
    f'  Revenue growth: {_safe_pct(revenue_growth)} (YoY) · Earnings growth: {_safe_pct(earnings_growth)} (YoY — <5% is flat, not growing)',
    f'  EBITDA margin: {_safe_pct(ebitda_margins)} · Net margin: {_safe_pct(profit_margin)} · ROA: {_safe_pct(return_on_assets)}',
    f'  OCF quality: {_safe_num(ocf_quality, 2)} · Free cash flow: {"positive" if free_cashflow and free_cashflow > 0 else "negative" if free_cashflow and free_cashflow < 0 else "N/A"}',
    f'  P/E: {_safe_num(trailing_pe)} · PEG: {_safe_num(peg_ratio, 2)} · D/E: {_safe_num(debt_to_equity)}% · Current ratio: {_safe_num(current_ratio, 2)}',
    (f'  Interest coverage: {_safe_num(interest_coverage, 1)}x' if interest_coverage else '  Interest coverage: N/A'),
    (f'  Analyst: {analyst_label} ({analyst_count} analysts) · Direction: {analyst_direction_label} · Upside to mean: {analyst_upside:.1f}%'
     if analyst_count and analyst_upside else f'  Analyst: {analyst_label if analyst_label else "N/A"}'),
    '',
    'India-specific:',
    f'  Promoter holding: {_safe_pct(promoter_stake)} · Institutional: {_safe_pct(inst_holding)}',
    (f'  Promoter pledge: {promoter_pledge_pct:.1f}% ({"minimal — no stress" if promoter_pledge_pct < 5 else "HIGH — forced-sell risk" if promoter_pledge_pct > 25 else "moderate"})'
     if globals().get('promoter_pledge_pct') is not None else '  Promoter pledge: unavailable'),
    (f'  Stock PCR: {stock_pcr:.2f} ({"bullish — calls > puts" if stock_pcr < 0.7 else "bearish — puts > calls" if stock_pcr > 1.2 else "neutral"})'
     if globals().get('stock_pcr') is not None else '  Stock PCR: unavailable'),
    '',
    'Corporate events:',
    corporate_context,
    '',
    'Recent news:',
    news_context,
]

# Append macro context if cell-macro was run
_mc = globals().get('macro_context', {})
if _mc:
    lines += ['', 'Market context (4 PM post-close):']
    if _mc.get('nifty_return_pct') is not None:
        lines.append(f'  Nifty today: {_mc["nifty_return_pct"]:+.2f}%')
    if _mc.get('sector_return_pct') is not None:
        lines.append(f'  {_mc["sector_name"]}: {_mc["sector_return_pct"]:+.2f}%')
    if _mc.get('india_vix'):
        lines.append(f'  India VIX: {_mc["india_vix"]:.1f}')
    if _mc.get('fii_net_cr') is not None:
        lines.append(f'  FII net: ₹{_mc["fii_net_cr"]:,.0f} Cr ({"buying" if _mc["fii_net_cr"] >= 0 else "selling"})')
    if _mc.get('brent_price'):
        lines.append(f'  Brent: ${_mc["brent_price"]:.1f}')
    if _mc.get('usdinr'):
        lines.append(f'  USD/INR: ₹{_mc["usdinr"]:.2f}')

indicators_summary = '\n'.join(lines)

# ── System prompt ─────────────────────────────────────────────────────────────
system_prompt = """You are a stock analyst writing plain-English verdicts for Indian retail investors on NSE.

SCORING: 31 rule-based indicator votes across 3 categories — 12 technical, 15 fundamental, 4 India-specific. Each category is scored from -1 to +1 and averaged equally (33/33/33). The combined score drives the signal.

India-specific signals matter: promoter pledging >25% is a serious red flag (forced selling risk if stock falls). Stock PCR <0.7 = bullish positioning. Institutional holdings >30% = smart money present.

Return a JSON object with exactly these fields: signal, confidence, entry, stop_loss, target, whats_happening, why_it_matters, watch_out_for, trader_action, investor_action.

FIELD TYPES — strict, no exceptions:
  signal: string — exactly one of "BUY", "SELL", "HOLD"
  confidence: float between 0.0 and 1.0
  entry: integer — plain whole number only, no ₹ symbol, no commas, no ranges, no strings. Single mid-point price. Example: 1350
  stop_loss: integer — same rules. 0 if HOLD.
  target: integer — same rules. 0 if HOLD.

SIGNAL RULES:
  STRONG BUY → BUY  |  LEAN BUY → BUY  |  NEUTRAL → HOLD  |  LEAN SELL → SELL  |  STRONG SELL → SELL

CONFIDENCE — start from base, subtract reductions, apply floor:
  STRONG BUY/SELL base: 0.80 | LEAN BUY/SELL base: 0.62 | NEUTRAL base: 0.40
  − Categories strongly disagree (any two differ by >0.4): −0.10
  − Tech and fundamental scores differ by >0.4: −0.08 (additional)
  − Earnings within 5 days: −0.18
  − Serious negative news (probe, fraud, management crisis): −0.10
  − Low delivery % (<30%, speculative): −0.05
  − High promoter pledge (>25%): −0.08
  Floor: 0.30 minimum. If final confidence <0.40 for BUY/SELL → change signal to HOLD.

ENTRY / STOP / TARGET:
  HOLD: entry = current price as integer, stop_loss = 0, target = 0.
  BUY/SELL: entry = single integer (mid-point of any range you'd consider), stop 5–8% from entry, target 8–15% from entry.

LANGUAGE RULES — strict, no exceptions:
  Rule 1 — NO indicator names. Never write: MACD, RSI, Bollinger, Stochastic, moving average, EMA, SMA, CCI, Williams, crossover.
  Rule 2 — Match language to actual numbers. Earnings growth <5% is flat. Never write "improving earnings" for sub-5% growth.
    0–5%: flat / barely growing | 5–15%: growing modestly | >15%: growing strongly
  Rule 3 — If fundamentals look strong because of analyst consensus but earnings are weak, say so explicitly.
  Rule 4 — No empty hedging. Never write: "it is important to note", "investors should be aware", "depending on market conditions".

SECTIONS:
  whats_happening  — 2 sentences. What is the price doing and why in plain language. No indicator names.
  why_it_matters   — 2 sentences. What do fundamentals actually say. Name the tension if tech and fundamental disagree.
  watch_out_for    — 1–2 sentences. Real risks only: earnings date, pledging risk, legal probe, macro headwinds.
  trader_action    — 2 sentences for short-term traders (days to weeks).
    If BUY/SELL: "Enter around ₹[entry]. Stop at ₹[stop]. Target ₹[target] in [timeframe]."
    If HOLD: "No trade setup — [specific reason why direction is unclear]."
  investor_action  — 2 sentences for long-term investors (months to years).
    No stop-loss concept — investors accumulate on dips or trim on overvaluation."""

response = openai_client.chat.completions.create(
    model='gpt-5.4-mini',
    response_format={'type': 'json_object'},
    messages=[
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': indicators_summary},
    ],
)

verdict = json.loads(response.choices[0].message.content)
usage   = response.usage
print(f'Tokens — input: {usage.prompt_tokens}, output: {usage.completion_tokens}')
print('Raw verdict:')
print(json.dumps(verdict, indent=2))

Tokens — input: 1757, output: 322
Raw verdict:
{
  "signal": "HOLD",
  "confidence": 0.4,
  "entry": 1362,
  "stop_loss": 0,
  "target": 0,
  "whats_happening": "The stock is holding near \u20b91,362 after a modest run from the 52-week low, but the short-term picture is not clean because volume is weak and the broader market was softer today. A low-conviction bullish crossover is being outweighed by a long-term downtrend, so the price action looks mixed rather than decisive.",
  "why_it_matters": "The business picture is still decent: revenue is growing modestly, margins are healthy, cash generation is positive, and analysts see meaningful upside. But earnings growth is basically flat at 0.6%, so the strong analyst view is ahead of the actual profit trend, while the recent probe headline adds a real overhang.",
  "watch_out_for": "Results are due in 1 day, so volatility can spike sharply around the announcement. The recent drone bribery probe headline may keep sentiment cautious even i

## Verdict

In [54]:
import re as _re

def _to_int(v):
    """Parse price field robustly — handles int, float, or string like '₹1,345–₹1,375'."""
    if isinstance(v, (int, float)):
        return int(v)
    nums = _re.findall(r'\d+', str(v).replace(',', ''))
    return int(nums[0]) if nums else 0

SEP = '━' * 64

signal          = verdict['signal']
confidence      = verdict['confidence']
entry           = _to_int(verdict['entry'])
stop_loss       = _to_int(verdict['stop_loss'])
target          = _to_int(verdict['target'])
whats_happening = verdict.get('whats_happening', '')
why_it_matters  = verdict.get('why_it_matters', '')
watch_out_for   = verdict.get('watch_out_for', '')
trader_action   = verdict.get('trader_action', '')
investor_action = verdict.get('investor_action', '')

signal_icon = {'BUY': '↑', 'SELL': '↓', 'HOLD': '→'}.get(signal, '→')

print(SEP)
print(f'{COMPANY_NAME.upper()}  ·  NSE: {SYMBOL.replace(".NS", "")}')
print(f'₹{current_price:,.0f}  ·  Signal: {signal} {signal_icon}  ·  Confidence: {confidence:.0%}  ·  Beta: {beta:.2f}' if beta else f'₹{current_price:,.0f}  ·  Signal: {signal} {signal_icon}  ·  Confidence: {confidence:.0%}')
print(SEP)

# 3-category vote tally
print(f'Score: {all_b} buy  ·  {all_n} neutral  ·  {all_s} sell  ({total_votes} indicators, 3-way equal weight)')
print(f'  Technical      ({len(votes):2d}): {tech_grade:12s}  {tb}B · {tn}N · {ts}S')
print(f'  Fundamental    ({len(fund_votes):2d}): {fund_grade:12s}  {fb}B · {fn}N · {fs}S')
print(f'  India-specific ({len(india_votes):2d}): {india_grade:12s}  {ib}B · {in_}N · {is_}S')
print()

if signal == 'HOLD':
    print('  No trade. Direction is unclear — wait for a clearer setup.')
else:
    print(f'  Entry ₹{entry:,}  ·  Stop ₹{stop_loss:,}  ·  Target ₹{target:,}')
print()
print(SEP)
print()

if whats_happening:
    print('What\'s Happening')
    print(whats_happening)
    print()

if why_it_matters:
    print('Why It Matters')
    print(why_it_matters)
    print()

if watch_out_for:
    print('Watch Out For')
    print(watch_out_for)
    print()

print(SEP)
print()

if trader_action:
    print('For Traders  (days to weeks)')
    print(trader_action)
    print()

if investor_action:
    print('For Investors  (months to years)')
    print(investor_action)
    print()

print(SEP)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RELIANCE INDUSTRIES  ·  NSE: RELIANCE
₹1,362  ·  Signal: HOLD →  ·  Confidence: 40%  ·  Beta: 0.22
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Score: 10 buy  ·  17 neutral  ·  4 sell  (31 indicators, 3-way equal weight)
  Technical      (12): LEAN SELL     2B · 6N · 4S
  Fundamental    (15): STRONG BUY    7B · 8N · 0S
  India-specific ( 4): LEAN BUY      1B · 3N · 0S

  No trade. Direction is unclear — wait for a clearer setup.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

What's Happening
The stock is holding near ₹1,362 after a modest run from the 52-week low, but the short-term picture is not clean because volume is weak and the broader market was softer today. A low-conviction bullish crossover is being outweighed by a long-term downtrend, so the price action looks mixed rather than decisive.

Why It Matters
The business picture is still decent: revenue is growing modestly, ma

---

**Stock analysis complete.** Note the token counts, latencies, and whether the verdict reflects the actual market situation before iterating further.

---

## Global Cues  ·  8 AM Pre-Open

Run at **8 AM IST**, independently of the 4 PM analysis. US market closed ~1:30 AM IST — full data available. Asian markets are just opening.

**Time logic:**
- **4 PM stock analysis** — compute signal from settled NSE data, store verdict
- **8 AM global cues** — check if overnight US/Asia changed the risk picture before acting

GIFT Nifty is the single best predictor of NSE's opening direction. S&P 500 futures are the best forward indicator for tonight's US session.

In [55]:
# ── 8 AM Global Cues ────────────────────────────────────────────────────────
# Run at 8 AM IST — before NSE opens at 9:15 AM.
# US closed ~1:30 AM IST (full close available). Asian markets are opening.
# S&P futures are live — best forward indicator for tonight's US session.
# Standalone — only needs SYMBOL and COMPANY_NAME from cell-setup.

now_ist = datetime.now(IST)

def _gq(sym, period='5d'):
    """Fetch yfinance history, return (last_price, day_return_pct)."""
    try:
        h = yf.Ticker(sym).history(period=period)
        if h.empty: return None, None
        price = h['Close'].iloc[-1]
        ret   = (h['Close'].iloc[-1] - h['Close'].iloc[-2]) / h['Close'].iloc[-2] * 100 if len(h) >= 2 else None
        return price, ret
    except Exception:
        return None, None

def _fmt_line(label, price, ret, prefix='', suffix=''):
    if price is None: return f'  {label:14s}: N/A'
    ret_str = f'  {("↑" if ret >= 0 else "↓")}{abs(ret):.2f}%' if ret is not None else ''
    return f'  {label:14s}: {prefix}{price:,.1f}{ret_str}{suffix}'

# US close (last night — fully available at 8 AM IST)
sp_price,  sp_ret  = _gq('^GSPC')
nq_price,  nq_ret  = _gq('^IXIC')
dji_price, dji_ret = _gq('^DJI')

# Asian markets (opening / early session at 8 AM IST)
nk_price,  nk_ret  = _gq('^N225')   # Nikkei — opens 5:30 AM IST
hsi_price, hsi_ret = _gq('^HSI')    # Hang Seng — opens 7:00 AM IST

# Forward-looking: S&P 500 futures (live 24h — directional hint for tonight)
esf_price, esf_ret = _gq('ES=F')

# Commodities (24h — always live)
brent_price, brent_ret = _gq('BZ=F')
gold_price,  gold_ret  = _gq('GC=F')
usdinr,      _         = _gq('INR=X')

# ── Display ──────────────────────────────────────────────────────────────────
SEP = '━' * 64
print(SEP)
print(f'GLOBAL CUES  ·  {now_ist.strftime("%d %b %Y  %H:%M IST")}  ·  Run at 8 AM pre-open')
print(SEP)

print('US CLOSE  (last night)')
print(_fmt_line('S&P 500',   sp_price,  sp_ret))
print(_fmt_line('Nasdaq',    nq_price,  nq_ret))
print(_fmt_line('Dow Jones', dji_price, dji_ret))
print()

print('ASIA  (opening / early session)')
print(_fmt_line('Nikkei',    nk_price,  nk_ret))
print(_fmt_line('Hang Seng', hsi_price, hsi_ret))
if esf_price:
    print(_fmt_line('S&P futures', esf_price, esf_ret, suffix='  ← tonight\'s US direction'))
print()

print('COMMODITIES')
if brent_price:
    print(_fmt_line('Brent crude', brent_price, brent_ret, prefix='$'))
if gold_price:
    print(_fmt_line('Gold',        gold_price,  gold_ret,  prefix='$'))
if usdinr:
    print(f'  {"USD/INR":14s}: ₹{usdinr:.2f}')
print()

# Rule-based global read
reads = []
if sp_ret is not None:
    if sp_ret < -1:  reads.append(f'US fell {abs(sp_ret):.1f}% — gap-down open likely')
    elif sp_ret > 1: reads.append(f'US rose {sp_ret:.1f}% — positive carry-over expected')
if nk_ret  is not None and nk_ret  < -1: reads.append('Nikkei weak — Asia risk-off')
if hsi_ret is not None and hsi_ret < -1: reads.append('Hang Seng falling — China/HK risk-off')
if brent_ret is not None and brent_ret > 2: reads.append(f'oil up {brent_ret:.1f}% — inflation concern')
if esf_ret is not None:
    if esf_ret >  0.5: reads.append('S&P futures positive — US likely strong tonight')
    elif esf_ret < -0.5: reads.append('S&P futures negative — potential headwind tonight')

print('Global read  :  ' + (' · '.join(reads).capitalize() if reads else 'Neutral — no strong overnight signal'))
print(SEP)

# ── Morning note: LLM update on yesterday's signal ───────────────────────────
# Only runs if the 4 PM verdict is available. Interprets overnight data through
# the lens of this specific stock and sector, not just generic market commentary.
_prev = globals().get('verdict')
if _prev:
    _sector = globals().get('macro_context', {}).get('sector_name', 'unknown sector')

    _overnight_lines = [
        f'Stock: {SYMBOL} ({COMPANY_NAME}) — Sector: {_sector}',
        f'',
        f'4 PM signal (computed yesterday):',
        f'  Signal: {_prev["signal"]}  Confidence: {_prev["confidence"]:.0%}',
        f'  Entry ₹{int(_prev["entry"]):,}  Stop ₹{int(_prev["stop_loss"]):,}  Target ₹{int(_prev["target"]):,}',
        f'',
        f'Overnight developments (since 4 PM yesterday):',
    ]
    if sp_ret  is not None: _overnight_lines.append(f'  US S&P 500: {sp_ret:+.2f}%  Nasdaq: {nq_ret:+.2f}%' if nq_ret else f'  US S&P 500: {sp_ret:+.2f}%')
    if nk_ret  is not None: _overnight_lines.append(f'  Nikkei: {nk_ret:+.2f}%')
    if hsi_ret is not None: _overnight_lines.append(f'  Hang Seng: {hsi_ret:+.2f}%')
    if esf_ret is not None: _overnight_lines.append(f'  S&P futures: {esf_ret:+.2f}% (live)')
    if brent_ret is not None: _overnight_lines.append(f'  Brent crude: ${brent_price:.1f}  {brent_ret:+.2f}%')
    if gold_ret  is not None: _overnight_lines.append(f'  Gold: ${gold_price:,.0f}  {gold_ret:+.2f}%')
    if usdinr:  _overnight_lines.append(f'  USD/INR: ₹{usdinr:.2f}')

    _morning_prompt = '\n'.join(_overnight_lines)

    _morning_system = f"""You are a pre-market analyst writing a brief morning update for Indian retail investors.

Yesterday's 4 PM signal has already been computed for this stock using 31 rule-based indicators.
Your job is NOT to re-analyze the stock. Your job is to answer one question:

Given this specific stock's business and sector, do the overnight developments SUPPORT, WEAKEN, or NEUTRALIZE the {_prev["signal"]} signal from yesterday?

Rules:
- Be stock-specific. The same US move affects an IT exporter differently from an energy or banking stock.
- USD/INR weakening helps IT exporters, hurts oil importers.
- Oil price moves matter most for energy/refining stocks, airlines, paint companies.
- US tech rally benefits Indian IT sentiment.
- Weak Asian markets (Nikkei/Hang Seng) signal broad risk-off — matters more for export-heavy sectors.
- If the signal was HOLD, say whether overnight developments make a setup more or less likely today.

Write exactly 2–3 sentences. Plain English. No indicator names. End with one clear verdict:
"Signal INTACT", "Signal WEAKENED", or "Signal STRENGTHENED"."""

    _mr = openai_client.chat.completions.create(
        model='gpt-5.4-mini',
        messages=[
            {'role': 'system', 'content': _morning_system},
            {'role': 'user',   'content': _morning_prompt},
        ],
    )
    morning_note = _mr.choices[0].message.content.strip()
    _mu = _mr.usage
    print()
    print(f'MORNING UPDATE  ·  {COMPANY_NAME} ({_prev["signal"]} from yesterday)')
    print('─' * 64)
    print(morning_note)
    print(f'─' * 64)
    print(f'Tokens — input: {_mu.prompt_tokens}, output: {_mu.completion_tokens}')
else:
    print()
    print(f'Context: analysis signal was computed at 4 PM yesterday for {COMPANY_NAME}.')
    print('Run the 4 PM cells first, then re-run this cell to get the morning update.')
print()
print('NSE opens at 9:15 AM IST — check if overnight developments change the risk/reward before acting.')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GLOBAL CUES  ·  23 Apr 2026  08:16 IST  ·  Run at 8 AM pre-open
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
US CLOSE  (last night)
  S&P 500       : 7,137.9  ↑1.05%
  Nasdaq        : 24,657.6  ↑1.64%
  Dow Jones     : 49,490.0  ↑0.69%

ASIA  (opening / early session)
  Nikkei        : 58,952.1  ↓1.06%
  Hang Seng     : 25,928.5  ↓0.90%
  S&P futures   : 7,138.0  ↑0.54%  ← tonight's US direction

COMMODITIES
  Brent crude   : $103.0  ↑4.55%
  Gold          : $4,729.4  ↑0.66%
  USD/INR       : ₹93.78

Global read  :  Us rose 1.0% — positive carry-over expected · nikkei weak — asia risk-off · oil up 4.5% — inflation concern · s&p futures positive — us likely strong tonight
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

MORNING UPDATE  ·  Reliance Industries (HOLD from yesterday)
────────────────────────────────────────────────────────────────
Reliance gets a clear tailwind from the sha